In [ ]:
## -----Iniciate Data----- ##

## Import necessary bib ##
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pypsa
from datetime import datetime
pip install adjustText
from adjustText import adjust_text
import pandas as pd
import numpy as np
import os

## load network ##
# choose network
#file_apollo = "/home/pypsa_eur/work/pypsa-eur-evt/basemap_V4.nc"
file_apollo = "/home/pypsa_eur/work/pypsa-eur-evt/results/48_wAT_entsoe_2030NT_apollo_fa_sp/networks/base_s_adm_elec_.nc"

n = pypsa.Network()
n.import_from_netcdf(file_apollo)

ModuleNotFoundError: No module named 'adjustText'

In [2]:
## -----Saves Demand Series from Specific Nodes----- ##

# List of buses of interest
bus_names = ["ES00", "ITN1", "ITCN", "ITCS", "ITS1", "ITCA", "ITSI", "ITSA"]

# Initialize empty DataFrame to collect time series
all_demands = pd.DataFrame(index=n.snapshots)

for bus_name in bus_names:
    # Find all loads connected to this bus
    load_names = n.loads[n.loads.bus == bus_name].index

    # Get total demand time series for this bus
    demand_ts = n.loads_t.p_set[load_names].sum(axis=1)

    # Add to the DataFrame under the bus name
    all_demands[bus_name] = demand_ts

# Save the combined DataFrame to a single CSV file
all_demands.to_csv("/home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/demand_timeseries_selected_buses.csv")

print("Saved combined demand time series to 'demand_timeseries_selected_buses.csv'")

Saved combined demand time series to 'demand_timeseries_selected_buses.csv'


In [11]:
## -----Saving the Map with Line- and Link-Capacities----- ##

# ---- Load Your Network ----
network_path = "/home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/"  # Replace with your actual network file

# ---- Plotting Function ----
def plot_network(file_path, label):
    print(f"Processing: {label}")

    # ---- Shared Plotting Parameters ----
    projection_type = "LambertConformal"
    projection_params = {"central_longitude": 10, "central_latitude": 50}
    figsize = (8.27 * 2, 11.69 * 2)
    line_width = 0.5
    bus_size = 0.02
    bus_color = "cadetblue"
    line_color = "gray"
    link_color = "green"
    legend_loc = 'upper right'
    info_box_position = (0.95, 0.05)
    font_size_title = 14
    font_size_info = 8

    # Set up plot
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={
        "projection": getattr(ccrs, projection_type)(**projection_params)
    })

    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='lightblue')
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.COASTLINE)

    # Plot Buses
    ax.scatter(n.buses.x, n.buses.y, transform=ccrs.PlateCarree(), s=bus_size * 1000,
               color=bus_color, zorder=3)

    for i, bus in n.buses.iterrows():
        if bus.carrier in ["AC", "DC"]:
            ax.text(bus.x, bus.y + 0.2, str(i), transform=ccrs.PlateCarree(),
                    fontsize=4, ha='center', va='center', color='black',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.6))

    # AC Lines
    texts = []
    for i, line in n.lines.iterrows():
        if line.carrier == "AC":
            x0, y0 = n.buses.loc[line.bus0, ['x', 'y']]
            x1, y1 = n.buses.loc[line.bus1, ['x', 'y']]
            xm, ym = (x0 + x1) / 2, (y0 + y1) / 2

            ax.plot([x0, x1], [y0, y1], transform=ccrs.PlateCarree(),
                    color=line_color, linewidth=line_width, zorder=2)

            texts.append(
                ax.text(xm, ym, f"{line.s_nom:.1f} MVA", transform=ccrs.PlateCarree(),
                        fontsize=5, ha='center', va='center', color='gray',
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.5))
            )

    # DC Links
    for i, link in n.links.iterrows():
        if link.carrier == "DC":
            x0, y0 = n.buses.loc[link.bus0, ['x', 'y']]
            x1, y1 = n.buses.loc[link.bus1, ['x', 'y']]
            xm, ym = (x0 + x1) / 2, (y0 + y1) / 2

            ax.plot([x0, x1], [y0, y1], transform=ccrs.PlateCarree(),
                    color=link_color, linewidth=line_width, linestyle="--", zorder=2)

            ax.text(xm, ym, f"{link.p_nom:.1f} MVA", transform=ccrs.PlateCarree(),
                    fontsize=5, ha='center', va='bottom', color='green',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.5))

    # Adjust label placement
    adjust_text(texts, ax=ax, expand_text=(1.05, 1.2), arrowprops=dict(arrowstyle='-', color='black', lw=0.3))

    # Legend
    transmission_legend = mlines.Line2D([], [], color=line_color, linewidth=line_width, label="HVAC Transmission Lines")
    link_legend = mlines.Line2D([], [], color=link_color, linewidth=line_width, linestyle="--", label="HVDC Transmission Lines")
    bus_legend = mlines.Line2D([], [], color=bus_color, marker='o', linestyle='None', markersize=6, label="Buses")
    ax.legend(handles=[transmission_legend, link_legend, bus_legend], loc=legend_loc)

    # Title and Info Box
    ax.set_title(f"ENTSOE 128 Knoten – {label}", fontsize=font_size_title, loc='center', pad=20)
    date_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    info_text = f"Projection: {projection_type}\nDate: {date_str}\nPyPSA-Eur"
    ax.text(info_box_position[0], info_box_position[1], info_text, transform=ax.transAxes, fontsize=font_size_info,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

    # Save Plot
    filename = f"/home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/network{label.replace(' ', '_')}.pdf"
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    plt.savefig(filename, format='pdf', bbox_inches='tight', dpi=1000)
    plt.close()
    print(f"Saved: {filename}")

# ---- Call the Function ----
plot_network(network_path, "_electricity")

Processing: _electricity
Saved: /home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/network_electricity.pdf


In [ ]:
## -----Save all Lines and Links and their Properities in a csv-file----- ##

# Nur Links mit carrier == "DC"
dc_links = n.links[n.links.carrier == "DC"]

# Excel-Datei speichern
output_file_links = "/home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/links_dc_p_nom_export.xlsx"
dc_links.to_excel(output_file_links, index=True)

print(f"DC-Links gespeichert unter: {output_file_links}")

# Nur Leitungen mit carrier == "AC"
ac_lines = n.lines[n.lines.carrier == "AC"]

# Excel-Datei speichern
output_file_lines = "/home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/lines_ac_s_nom_export.xlsx"
ac_lines.to_excel(output_file_lines, index=True)

print(f"AC-Leitungen gespeichert unter: {output_file_lines}")

DC-Links gespeichert unter: links_dc_p_nom_export.xlsx
AC-Leitungen gespeichert unter: /home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/lines_ac_s_nom_export.xlsx


In [ ]:
## -----Apollo-link Calculations----- ##

link_name = "TYNDP2024_1210"

price_ES = n.buses_t.marginal_price["ES00"]
price_IT = n.buses_t.marginal_price["ITN1"]
power_flow = n.links_t.p0[link_name]
power_nom = n.links.p_nom[link_name]

def calculate_congestion_revenue(price_from, price_to, flow):
    """
    Berechnet absolute Engpasserlöse auf Basis von Preisunterschieden und Flüssen.
    Gibt Gesamterlös, stundenweise Erlöse und Preisunterschiede zurück.
    """
    price_from = np.array(price_from)
    price_to = np.array(price_to)
    flow = np.array(flow)

    price_diff = price_from - price_to

    revenue_per_hour = np.abs(price_diff * flow) * 8760 / len(price_diff) # Absoluter Erlös
    total_revenue = np.sum(revenue_per_hour)
    energy_transported = np.sum(flow) * (8760 / len(price_diff)) / 1000 # GWh

    return total_revenue, revenue_per_hour, price_diff, energy_transported

total_rev, rev_hourly, diff, energy_transported = calculate_congestion_revenue(price_ES, price_IT, power_flow)
full_load_hours = energy_transported / power_nom * 1000
print(f"🔢 Gesamt-Engpasserlös: {total_rev:,.2f} EUR")
print(f"🔢 Gesamt transportierte Energiemenge: {energy_transported:,.2f} GWh")
print(f"🔢 Volllaststunden Leitung: {full_load_hours:,.2f} h")

# DataFrame erstellen
df = pd.DataFrame({
    "price_from": price_ES,
    "price_to": price_IT,
    "flow": power_flow
})

# Optional: Index als Spalte (Zeit)
df["timestamp"] = df.index

# Spaltenreihenfolge anpassen
df = df[["timestamp", "price_from", "price_to", "flow"]]

# CSV exportieren
csv_path = "/home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/congestion_timeseries.csv"
df.to_csv(csv_path, index=False)

print(f"✅ CSV gespeichert unter: {csv_path}")

🔢 Gesamt-Engpasserlös: 289,324,519.30 EUR
🔢 Gesamt transportierte Energiemenge: 18,210.81 GWh
🔢 Volllaststunden Leitung: 8,709.14 h
✅ CSV gespeichert unter: /home/pypsa_eur/work/pypsa-eur-evt/notebooks-evt/TYNDP_output/congestion_timeseries.csv
